# 01 – 下载、清洗、切分、段落编号
运行顺序：从上到下。raw 文件只下载一次（遵守 Gutenberg robot access 政策），之后读本地文件。
输出：`data/processed/<story_id>.txt`，每段形如 `[0001] ...`，以及 `manifest/annotations.csv` 模板。

In [ ]:
import os, re, time, urllib.request, pandas as pd
from pathlib import Path
# Work from the repo root (the folder holding CLAUDE.md) even if the kernel starts in notebooks/,
# so data/ and manifest/ resolve to the top-level folders.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
os.chdir(ROOT)
print("repo root:", ROOT)
for d in ["data/raw","data/processed","manifest"]: os.makedirs(d, exist_ok=True)

RAW = "data/raw/adventures.txt"
URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"   # The Adventures of Sherlock Holmes
if not os.path.exists(RAW):
    req = urllib.request.Request(URL, headers={"User-Agent": "course-project (yw5688@nyu.edu)"})
    open(RAW, "wb").write(urllib.request.urlopen(req).read()); time.sleep(2)
raw = open(RAW, encoding="utf-8-sig").read().replace("\r\n", "\n")
print(len(raw), "chars")

In [ ]:
# 去掉 Gutenberg 头尾
s = raw.find("*** START OF"); s = raw.find("\n", s) + 1
e = raw.find("*** END OF")
body = raw[s:e]

# Story heading lines look like 'I. A SCANDAL IN BOHEMIA' (no 'ADVENTURE' prefix in the current pg1661).
# Requiring an all-caps title at column 0 skips the indented mixed-case table of contents
# and the bare section numbers ('I.', 'II.') inside stories. Keep the last hit per numeral as a safeguard.
pat = re.compile(r"^([IVX]+)\.\s+([A-Z][A-Z’'\- ]+?)\s*$", re.M)
hits = {}
for m in pat.finditer(body):
    hits[m.group(1)] = (m.start(), m.end(), m.group(2).title())
starts = sorted(hits.values())
print(len(starts), "stories found (expect 12)")
for st in starts: print(st[2], st[0])

In [ ]:
def slugify(t): return re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")

stories = {}
for i, (a, b, title) in enumerate(starts):
    end = starts[i+1][0] if i+1 < len(starts) else len(body)
    chunk = body[b:end]
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", chunk)]
    paras = [p for p in paras if p]
    # 章节小标题（如 'I.' 'II.' 或全大写短行）单独成段，保留但标注时可忽略
    stories[title] = paras

for t, ps in stories.items():
    print(f"{t:45s} {len(ps):4d} paragraphs | first: {ps[0][:50]!r} | last: {ps[-1][-50:]!r}")

In [ ]:
# ★ 人工检查：上面每篇的 first/last 是否是故事真正的开头/结尾？确认后再写文件
PILOT = {  # story_id -> 标题（需与上面 title 完全一致，大小写不敏感）
 "P1": "The Adventure Of The Speckled Band",
 "P2": "The Red-Headed League",
 "P3": "The Adventure Of The Blue Carbuncle",
 "P4": "A Case Of Identity",
 "P5": "The Adventure Of The Copper Beeches",
}
norm = lambda t: re.sub(r"[^a-z]", "", t.lower())
by_norm = {norm(t): t for t in stories}
for sid, title in PILOT.items():
    t = by_norm[norm(title)]
    with open(f"data/processed/{sid}.txt", "w", encoding="utf-8") as f:
        for i, p in enumerate(stories[t], start=1):
            f.write(f"[{i:04d}] {p}\n\n")
    print(sid, t, len(stories[t]))

In [ ]:
cols = ["story_id","title","annotator","culprit","reveal_para_idx","reveal_pct","reveal_quote",
        "aliases","culprit_before_reveal","exclusion_reason","notes"]
if not os.path.exists("manifest/annotations.csv"):
    pd.DataFrame([{"story_id":k,"title":v} for k,v in PILOT.items()], columns=cols)\
      .to_csv("manifest/annotations.csv", index=False)

# 标完后运行：reveal_pct = reveal_para_idx / 总段数
def fill_pct(path="manifest/annotations.csv"):
    m = pd.read_csv(path)
    for i, r in m.iterrows():
        if pd.notna(r.reveal_para_idx):
            n = sum(1 for l in open(f"data/processed/{r.story_id}.txt", encoding="utf-8") if l.startswith("["))
            m.loc[i, "reveal_pct"] = round(r.reveal_para_idx / n, 3)
    m.to_csv(path, index=False)